# ViHalluMT — Sinh ngữ liệu tiếng Việt (Kaggle / Colab)Notebook này chạy phần **cần GPU** của đồ án: dịch câu nguồn bằng nhiều hệ dịchvà nhiều cấu hình giải mã, chấm điểm sơ bộ, rồi lấy mẫu phân tầng ra tập cầngán nhãn tay.**Đầu ra:**- `candidates.jsonl` — toàn bộ ứng viên kèm điểm- `to_annotate.jsonl` — mẫu đã chọn, tải về máy để gán nhãn- `corpus_a_stats.csv` — thống kê cho báo cáo**Thời gian ước tính:** 30–50 phút trên 1×T4 với 3.000 câu nguồn mỗi hướng.> **Bật GPU trước khi chạy:** Kaggle → *Settings → Accelerator → GPU T4 x2*.> Colab → *Runtime → Change runtime type → T4 GPU*.

## 1. Cài đặt thư viện và cấu hình môi trường

In [ ]:
import subprocess, sys, osdef sh(cmd):    print(f"$ {cmd}")    subprocess.run(cmd, shell=True, check=False)sh(f"{sys.executable} -m pip install -q -U transformers accelerate sentencepiece")sh(f"{sys.executable} -m pip install -q sentence-transformers datasets underthesea")

In [ ]:
import torch, transformersprint("torch       :", torch.__version__)print("transformers:", transformers.__version__)print("CUDA        :", torch.cuda.is_available())if torch.cuda.is_available():    for i in range(torch.cuda.device_count()):        p = torch.cuda.get_device_properties(i)        print(f"  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB")else:    print("  !! Chua bat GPU — buoc dich se rat cham. Hay bat GPU roi chay lai.")

### Lấy mã nguồnChọn **một** trong hai cách:**Cách A — clone từ GitHub.** `REPO_URL` đã điền sẵn kho của đồ án.> Kho đang để **private**, nên `git clone` trên Kaggle sẽ treo vì đòi mật khẩu.> Trước khi chạy, tạm chuyển sang public:> ```bash> gh repo edit justinnguyendsa/vihallumt --visibility public --accept-visibility-change-consequences> ```> Chạy xong thì chuyển lại private bằng lệnh tương tự với `--visibility private`.> Không muốn đổi thì dùng **Cách B**.**Cách B — tải lên Kaggle Dataset.** Nén thư mục `src/` và `scripts/` thành`.zip`, tạo một Kaggle Dataset mới, rồi *Add Data* vào notebook này. Sau đó sửa`LOCAL_SRC` cho trỏ đúng đường dẫn.

In [ ]:
REPO_URL  = "https://github.com/justinnguyendsa/vihallumt.git"LOCAL_SRC = "/kaggle/input/vihallumt-src"                     # dung cho Cach Bimport os, sys, shutilfrom pathlib import PathWORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()PROJECT = WORK / "vihallumt"if REPO_URL and "<" not in REPO_URL:    if PROJECT.exists():        shutil.rmtree(PROJECT)    os.system(f"git clone --depth 1 {REPO_URL} {PROJECT}")elif Path(LOCAL_SRC).exists():    PROJECT.mkdir(parents=True, exist_ok=True)    for sub in ("src", "scripts"):        if (Path(LOCAL_SRC) / sub).exists():            shutil.copytree(Path(LOCAL_SRC) / sub, PROJECT / sub, dirs_exist_ok=True)else:    raise SystemExit(        "Chua co ma nguon. Hay dat REPO_URL (Cach A) hoac them Kaggle Dataset (Cach B)."    )sys.path.insert(0, str(PROJECT / "src"))os.chdir(PROJECT)print("Thu muc lam viec:", Path.cwd())print("Cac tep:", sorted(p.name for p in Path.cwd().iterdir()))

In [ ]:
# Kiem tra import duoc goi vihallumtfrom vihallumt.corpus.translate import DEFAULT_GENERATION_PLAN, probe_translatorfrom vihallumt.corpus.sampling import stratified_sampleprint("Import OK. Ke hoach sinh du lieu:")for s in DEFAULT_GENERATION_PLAN:    print(f"  {s.tag:<28} {s.share:.0%}")

## 2. Kiểm tra các hệ dịch trước khi tốn GPUBước này chỉ nạp *tokenizer* (rẻ, nhanh) để phát hiện sớm mô hình không dùngđược. Thà biết ngay từ đầu còn hơn đổ sau 20 phút chạy GPU.`envit5` và `vinai-translate` dựa trên tokenizer sentencepiece và **vỡ vớitransformers ≥ 5.0**. Nếu chúng báo hỏng ở đây, script sẽ tự bỏ qua và chia lạitỉ trọng — nhớ ghi việc này vào mục *Limitations* của báo cáo.

In [ ]:
for name in sorted({s.translator for s in DEFAULT_GENERATION_PLAN}):    ok, msg = probe_translator(name)    print(f"  {name:<12} {'OK' if ok else 'HONG  ' + msg}")

## 3. (Tuỳ chọn) Mở khoá FLORES-200FLORES là dataset **gated**. Có nó thì kết quả tiếng Việt so sánh trực tiếpđược với 18 hướng dịch của HalOmi, vì HalOmi cũng lấy câu nguồn từ FLORES.1. Đồng ý điều khoản tại https://huggingface.co/datasets/openlanguagedata/flores_plus2. Tạo token tại https://huggingface.co/settings/tokens3. Trên Kaggle: *Add-ons → Secrets* → thêm secret tên `HF_TOKEN`Bỏ qua ô này cũng được — đường ống vẫn chạy với OPUS-100 + IWSLT.

In [ ]:
USE_FLORES = False   # <-- dat True sau khi da them HF_TOKENif USE_FLORES:    try:        from kaggle_secrets import UserSecretsClient        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")        print("Da nap HF_TOKEN tu Kaggle Secrets")    except Exception as e:        print("Khong lay duoc secret:", e)        print("Dat tay: os.environ['HF_TOKEN'] = '...'")

## 4. Sinh ngữ liệuChỉnh `N_SOURCE` và `N_ANNOTATE` cho khớp với sức gán nhãn của bạn.Mặc định 3.000 câu nguồn mỗi hướng → 700 cặp đưa đi gán nhãn.Muốn thử nhanh trước thì đặt `SMOKE = True` (chỉ vài phút).

In [ ]:
SMOKE      = FalseN_SOURCE   = 3000N_ANNOTATE = 700DIRECTIONS = "en-vi,vi-en"BATCH_SIZE = 32      # T4 chiu duoc 32; giam xuong neu bao het bo nhoargs = [    sys.executable, "scripts/build_corpus_a.py",    "--n-source", str(200 if SMOKE else N_SOURCE),    "--n-annotate", str(40 if SMOKE else N_ANNOTATE),    "--directions", DIRECTIONS,    "--batch-size", str(BATCH_SIZE),    "--plan", "smoke" if SMOKE else "default",]if not USE_FLORES:    args.append("--no-flores")print(" ".join(args), "\n")subprocess.run(args, check=True)

## 5. Kiểm tra kết quả

In [ ]:
import pandas as pdcand = pd.read_json("data/vihallumt/candidates.jsonl", lines=True)ann  = pd.read_json("data/vihallumt/to_annotate.jsonl", lines=True)print(f"Ung vien       : {len(cand)}")print(f"Cho gan nhan   : {len(ann)}")print()print("Theo huong dich:")print(cand.groupby(["direction", "perturbation"]).size().to_string())print()print("Theo he dich:")print(cand["gen_tag"].value_counts().to_string())print()print("Phan tang cua mau da chon:")print(ann.groupby("selection")[["agg_score"]].agg(["size", "mean"]).round(3).to_string())

In [ ]:
# Xem thu vai ban dich diem cao nhat — day la cac ca nghi ngo ao giactop = ann.nlargest(5, "agg_score")for _, r in top.iterrows():    print("-" * 78)    print("NGUON :", r["src_text_original"][:150])    print("DICH  :", r["mt_text"][:150])    print(f"        ({r['gen_tag']}, tang={r['selection']}, diem={r['agg_score']:.3f})")

In [ ]:
# Kiem tra suc khoe du lieu truoc khi dem di gan nhanissues = []if ann["mt_text"].str.strip().eq("").any():    issues.append("co ban dich rong")if ann["cand_id"].duplicated().any():    issues.append("co cand_id trung lap")if ann["selection"].nunique() < 3:    issues.append("thieu tang lay mau")if (ann["mt_text"] == ann["src_text_original"]).mean() > 0.1:    issues.append(">10% ban dich trung khit cau nguon (co the he dich khong dich)")print("VAN DE PHAT HIEN:", issues if issues else "khong co")

## 6. Tải kết quả về máyTrên Kaggle, mọi tệp trong `/kaggle/working` đều tải về được ở tab *Output* saukhi notebook chạy xong. Ô dưới chép kết quả ra đúng chỗ đó.Tải hai tệp này về, đặt vào `data/vihallumt/` trong kho mã nguồn ở máy bạn, rồichạy công cụ gán nhãn:```bashpython scripts/annotate.py```

In [ ]:
import shutilfrom pathlib import Pathdest = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()for name in ["data/vihallumt/candidates.jsonl",             "data/vihallumt/to_annotate.jsonl",             "results/corpus_a_stats.csv"]:    src = Path(name)    if src.exists():        target = dest / src.name        if src.resolve() != target.resolve():            shutil.copy(src, target)        print(f"  {target}  ({src.stat().st_size/1024:.0f} KB)")    else:        print(f"  !! thieu {name}")

## 7. Bước tiếp theo1. Tải `to_annotate.jsonl` về máy.2. Đọc kỹ [`docs/annotation-guideline-vi.md`](../docs/annotation-guideline-vi.md).3. Gán thử 50 cặp, chỉnh guideline theo ca khó gặp thực tế, rồi mới gán tiếp.4. `python scripts/annotate.py` — khoảng 150 cặp mỗi buổi.5. `python scripts/annotate.py --review` để xem thống kê nhãn đã gán.